# Check Drittauskunft correction algorithm

Goal: measure how the `correct_drittauskunft_prediction` algorithm behaves on real production predictions.

Steps:
1. Fetch `is_dritt = True` predictions (sample size = 1000) together with their cleaned texts (from `GPUTask`).
2. Apply the correction algorithm to each text.
3. Collect into a separate dataframe all cases where the LLM said *dritt* but the correction algorithm flipped it to *not dritt*.

In [2]:
import os
import sys
from configparser import RawConfigParser

import pandas as pd

# make the aftercourt_automation package importable (utils.*)
PROJECT_DIR = "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation"
if PROJECT_DIR not in sys.path:
    sys.path.append(PROJECT_DIR)

# make the intent_recognition package importable (src.* modules)
INTENT_RECOG_DIR = "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/intent_recognition"
if INTENT_RECOG_DIR not in sys.path:
    sys.path.append(INTENT_RECOG_DIR)

from src.db.database_manager import DatabaseManager, session_scope
from src.graph.repository.orm import GPUTask

# read the graph DB URI from secret.ini
_config = RawConfigParser()
_config.read(os.path.expanduser("~/secret.ini"))
GRAPH_DATABASE_URI = _config.get("GRAPH_DB", "GRAPH_DATABASE_URI")

graph_db_manager = DatabaseManager(db_uri=GRAPH_DATABASE_URI)


2026-06-17 16:24:43.756 | INFO     | src.db.database_manager:__init__:17 - Using provided DB URI.


In [3]:
# Correction algorithm (defined inline here so the notebook is self-contained and easy to iterate on).
import copy
import re

from loguru import logger

# ---- regexes ----------------------------------------------------------------
PROTOKOLL_DOC_REGEX = re.compile(
    r"(?im)^\s*(?:"
    r"protokoll"
    r"|verm[oö0]gens\s*auskunfts?\s*protokoll"
    r")\b"
)

ERGEBNIS_DOC_REGEX = re.compile(
    r"(?im)^\s*ergebnis(?:se)?(?:\s+der\s+verm[oö0]gens\s*auskunft)?\b"
)

DRITTAUSKUNFT_COVER_LETTER_REGEX = re.compile(
    r"(?is)(?:"
    r"drittausk[uü]nfte?\s+nach\s+§?\s*802\s*l\s*zpo\s+beim\s+"
    r"bundeszentralamt\s+f[üu]r\s+steuern\s+einzuholen"
    r"|das\s+ergebnis\s+teile\s+ich\s+ihnen\s+unter\s+[üu]bersendung"
    r"|in\s+der\s+anlage\s+das\s+ergebnis\s+der\s+auskunft\s+beim"
    r")"
)

DRITTAUSKUNFT_SECTIONS_REGEX = re.compile(
    r"(?im)^\s*(?:"
    r"name"
    r"|nachname"
    r"|vorname"
    r"|geburtsdatum"
    r"|kontonummer"
    r"|kontoinhaber"
    r"|verf[üu]gungsberechtigte"
    r")\s*:?",
)

# minimum number of drittauskunft result sections required to consider a
# cover-letter document a real drittauskunft
MIN_DRITTAUSKUNFT_SECTIONS = 3


# ---- correction function ----------------------------------------------------
def correct_drittauskunft_prediction(
    processed_model_output: dict | None,
    text: str | None,
    gpu_task_uuid: str | None = None,
) -> dict | None:
    """Precision-targeted correction for drittauskunft prediction based on document structure and keywords.

    Returns a corrected copy of ``processed_model_output`` (the input is not mutated).
    """
    if not processed_model_output or not text:
        return processed_model_output
    is_dritt = processed_model_output.get("is_dritt", False)
    if not is_dritt:
        return processed_model_output

    corrected = copy.deepcopy(processed_model_output)

    # Algorithm 1: protokoll without ergebnis -> not drittauskunft
    if PROTOKOLL_DOC_REGEX.search(text):
        if not ERGEBNIS_DOC_REGEX.search(text):
            corrected["is_dritt"] = False
            logger.info(
                "Drittauskunft prediction corrected to False via Algorithm 1 "
                "(Protokoll without Ergebnis): GPUTask[{}]",
                gpu_task_uuid,
            )
            return corrected

    # Algorithm 2: cover letter without ergebnis / too few sections -> not drittauskunft
    if DRITTAUSKUNFT_COVER_LETTER_REGEX.search(text):
        text = DRITTAUSKUNFT_COVER_LETTER_REGEX.sub("", text)
        if (
            not ERGEBNIS_DOC_REGEX.search(text)
            or len(DRITTAUSKUNFT_SECTIONS_REGEX.findall(text)) < MIN_DRITTAUSKUNFT_SECTIONS
        ):
            corrected["is_dritt"] = False
            logger.info(
                "Drittauskunft prediction corrected to False via Algorithm 2 "
                "(cover letter without Ergebnis or fewer than {} sections): GPUTask[{}]",
                MIN_DRITTAUSKUNFT_SECTIONS,
                gpu_task_uuid,
            )
            return corrected

    return corrected


In [10]:
# 1) Fetch is_dritt=True predictions (sample size = 1000) together with their texts.
SAMPLE_SIZE = 5000

# JSON filter: processed_model_output ->> 'is_dritt' == 'true'
is_dritt_filter = GPUTask.processed_model_output["is_dritt"].as_boolean() == True  # noqa: E712

with session_scope(graph_db_manager) as session:
    rows = (
        session.query(GPUTask)
        .filter(GPUTask.model_name == "drittauskunft_egvp")
        .filter(is_dritt_filter)
        .order_by(GPUTask.created_at.desc())
        .limit(SAMPLE_SIZE)
        .all()
    )
    dritt_df = pd.DataFrame(
        [{c.name: getattr(row, c.name) for c in GPUTask.__table__.columns} for row in rows]
    )

print("fetched rows:", len(dritt_df))
dritt_df.head()


fetched rows: 1103


,gpu_task_uuid,task_uuid,ticket_uuid,model_name,status,model_input,model_output,processed_model_output,retry_count,attachment_id,meta_info,created_at,updated_at
0,b6a22a61-c288-4a43-aca0-3a8773c18b09,2ed7c912-8a71-4127-bfc4-677a0996c39f,9497d89f-a02a-5a81-bfe7-870549197b2b,drittauskunft_egvp,done,{'text': 'Rebecca Bürger Hagener Straße 145 Ge...,"{'answer': '{""is_dritt"":true}', 'answer_reason...",{'is_dritt': True},0,68998773,"{'instance': {'private_ip': '100.2.36.225', 'i...",2026-06-17 13:24:52.412231,2026-06-17 13:41:26.732661
1,4b8b3a27-1a73-46fb-9276-3869472dc131,2da9e769-0dec-4804-a3ef-91b0e859d115,2caa308e-fc5f-5d77-999f-341f37b37d0e,drittauskunft_egvp,done,{'text': 'Dieses Dokument ist signiert mit Sig...,"{'answer': '{""is_dritt"":true}', 'answer_reason...",{'is_dritt': True},0,68998715,"{'instance': {'private_ip': '100.2.36.225', 'i...",2026-06-17 13:24:47.704690,2026-06-17 13:42:09.210428
2,fc97e50c-6fe4-4e4a-a38a-1781956dffd2,e9d93076-5edf-499b-a13b-ab51e5bb3b7b,beb1b8a1-61cd-52d4-999b-d85c29d385d4,drittauskunft_egvp,done,{'text': 'Hoffmann POSTANSCHRIFT: Postfach 03 ...,"{'answer': '{""is_dritt"":true}', 'answer_reason...",{'is_dritt': True},0,68998703,"{'instance': {'private_ip': '100.2.36.225', 'i...",2026-06-17 13:24:46.741358,2026-06-17 13:42:08.642688
3,4cad50ba-bd02-4feb-ab51-200e2e2523c2,e65606aa-091c-4d23-b128-e26d24d86b7d,79b0acfa-6f4d-5cd5-bb40-00012c835ee2,drittauskunft_egvp,done,{'text': 'J. Ohlandt Germaniapromenade 5 Oberg...,"{'answer': '{""is_dritt"":true}', 'answer_reason...",{'is_dritt': True},0,68998668,"{'instance': {'private_ip': '100.2.36.225', 'i...",2026-06-17 13:24:44.377711,2026-06-17 13:42:07.515992
4,56cb0507-b500-46ad-9a30-234f4ead3268,e2e9ec98-a9e9-4a06-b34a-47d62c0ce8f9,5baba24e-0be1-5386-889f-dfc756811640,drittauskunft_egvp,done,{'text': 'Bundeszentralamt für Steuern POSTANS...,"{'answer': '{""is_dritt"":true}', 'answer_reason...",{'is_dritt': True},0,68998535,"{'instance': {'private_ip': '100.2.36.225', 'i...",2026-06-17 13:24:32.652005,2026-06-17 13:41:43.270452


In [16]:
dritt_df.shape

(1103, 15)

In [11]:
# Extract the cleaned text used by the model (model_input['clean_text']).
def _extract_clean_text(model_input):
    if isinstance(model_input, dict):
        return model_input.get("clean_text") or ""
    return ""

dritt_df["text"] = dritt_df["model_input"].map(_extract_clean_text)

# sanity check: how many have non-empty text
print("rows with non-empty text:", (dritt_df["text"].str.len() > 0).sum())
dritt_df[["gpu_task_uuid", "attachment_id", "processed_model_output"]].head()


rows with non-empty text: 1103


,gpu_task_uuid,attachment_id,processed_model_output
0,b6a22a61-c288-4a43-aca0-3a8773c18b09,68998773,{'is_dritt': True}
1,4b8b3a27-1a73-46fb-9276-3869472dc131,68998715,{'is_dritt': True}
2,fc97e50c-6fe4-4e4a-a38a-1781956dffd2,68998703,{'is_dritt': True}
3,4cad50ba-bd02-4feb-ab51-200e2e2523c2,68998668,{'is_dritt': True}
4,56cb0507-b500-46ad-9a30-234f4ead3268,68998535,{'is_dritt': True}


In [12]:
# 2) Apply the correction algorithm to each text.
def _apply_correction(row):
    corrected = correct_drittauskunft_prediction(
        processed_model_output=row["processed_model_output"],
        text=row["text"],
        gpu_task_uuid=row["gpu_task_uuid"],
    )
    return bool(corrected.get("is_dritt", False)) if corrected else False

dritt_df["corrected_is_dritt"] = dritt_df.apply(_apply_correction, axis=1)

# True  -> still drittauskunft after correction
# False -> algorithm flipped it to NOT drittauskunft
dritt_df["corrected_is_dritt"].value_counts()


2026-06-17 16:27:08.084 | INFO     | __main__:correct_drittauskunft_prediction:67 - Drittauskunft prediction corrected to False via Algorithm 1 (Protokoll without Ergebnis): GPUTask[5bd4480e-d385-4a43-879d-d12177c4ddec]
2026-06-17 16:27:08.096 | INFO     | __main__:correct_drittauskunft_prediction:67 - Drittauskunft prediction corrected to False via Algorithm 1 (Protokoll without Ergebnis): GPUTask[aa358993-32be-47ee-8318-71d4a564a99c]
2026-06-17 16:27:08.115 | INFO     | __main__:correct_drittauskunft_prediction:67 - Drittauskunft prediction corrected to False via Algorithm 1 (Protokoll without Ergebnis): GPUTask[6038c453-4e2a-4cae-a4f8-367b50a40a01]
2026-06-17 16:27:08.138 | INFO     | __main__:correct_drittauskunft_prediction:67 - Drittauskunft prediction corrected to False via Algorithm 1 (Protokoll without Ergebnis): GPUTask[58c31f46-a1e3-4036-822b-7f5104d5e749]
2026-06-17 16:27:08.155 | INFO     | __main__:correct_drittauskunft_prediction:67 - Drittauskunft prediction corrected t

corrected_is_dritt
True     1082
False      21
Name: count, dtype: int64

In [13]:
# 3) Separate dataframe of cases the algorithm flipped: LLM said dritt, correction says NOT dritt.
corrected_df = dritt_df[~dritt_df["corrected_is_dritt"]].reset_index(drop=True)

n_total = len(dritt_df)
n_flipped = len(corrected_df)
print(f"LLM is_dritt=True samples : {n_total}")
print(f"flipped to NOT dritt      : {n_flipped} ({(n_flipped / n_total * 100) if n_total else 0:.1f}%)")

corrected_df[["gpu_task_uuid", "attachment_id", "created_at"]]


LLM is_dritt=True samples : 1103
flipped to NOT dritt      : 21 (1.9%)


,gpu_task_uuid,attachment_id,created_at
0,5bd4480e-d385-4a43-879d-d12177c4ddec,68998399,2026-06-17 13:22:43.425887
1,aa358993-32be-47ee-8318-71d4a564a99c,68989504,2026-06-17 08:25:54.012925
2,6038c453-4e2a-4cae-a4f8-367b50a40a01,68867765,2026-06-16 13:26:11.722312
3,58c31f46-a1e3-4036-822b-7f5104d5e749,68856476,2026-06-16 06:12:31.343785
4,fc3336a1-d574-43dd-bc13-e26ee0d2154e,68705724,2026-06-15 11:25:01.240759
5,b04e6677-d820-4295-a4df-ad56215ae674,68334592,2026-06-12 13:24:26.026738
6,4863b1fc-cb1d-4e5a-ba2b-d9af5e7c145a,68309623,2026-06-11 13:23:23.433344
7,c9e55333-7563-469d-9630-fc6491817275,68301088,2026-06-11 06:23:07.987766
8,ba28224a-a5de-40b1-9a2c-4717cfe24849,68260277,2026-06-10 20:14:57.371054
9,6d6c83fc-2d77-4e8d-a386-9fdfb516baa0,68260193,2026-06-10 20:14:50.809551


In [14]:
# Diagnostic: label which algorithm caused the flip (to see *how* the correction works).
def _flip_reason(text):
    if not text:
        return "no_text"
    # Algorithm 1: protokoll without ergebnis
    if PROTOKOLL_DOC_REGEX.search(text) and not ERGEBNIS_DOC_REGEX.search(text):
        return "algo1_protokoll_no_ergebnis"
    # Algorithm 2: cover letter without ergebnis / too few sections
    if DRITTAUSKUNFT_COVER_LETTER_REGEX.search(text):
        stripped = DRITTAUSKUNFT_COVER_LETTER_REGEX.sub("", text)
        n_sections = len(DRITTAUSKUNFT_SECTIONS_REGEX.findall(stripped))
        if not ERGEBNIS_DOC_REGEX.search(stripped):
            return "algo2_coverletter_no_ergebnis"
        if n_sections < MIN_DRITTAUSKUNFT_SECTIONS:
            return f"algo2_coverletter_few_sections({n_sections})"
    return "unknown"

corrected_df["flip_reason"] = corrected_df["text"].map(_flip_reason)
corrected_df["flip_reason"].value_counts()


flip_reason
algo1_protokoll_no_ergebnis      17
algo2_coverletter_no_ergebnis     4
Name: count, dtype: int64

In [15]:
# Inspect a single flipped case (change the index to review different documents).
idx = 0
if len(corrected_df):
    row = corrected_df.iloc[idx]
    print("gpu_task_uuid:", row["gpu_task_uuid"])
    print("attachment_id:", row["attachment_id"])
    print("flip_reason  :", row["flip_reason"])
    print("=" * 80)
    print(row["text"])
else:
    print("No flipped cases to inspect.")


gpu_task_uuid: 5bd4480e-d385-4a43-879d-d12177c4ddec
attachment_id: 68998399
flip_reason  : algo1_protokoll_no_ergebnis
Obergerichtsvollzieherin Astrid Weiß
Lusenstraße 3, 94161 Ruderting
beim Amtsgericht Passau
Bürozeiten:
Die & Do je von 9.00 Uhr -10.30 Uhr
Telefon:
08509/936158
Diensthandy: 016092806017
Email:
gvweiss@gmx.de
Dienstkonto:
DE64 740 627 860
BIC: GENODEF1TIE
OGVin A. Weiß, Lusenstraße 3, 94161 Ruderting
Raiffeisenbank i.Lkrs. Passau-Nord eG
EGV-Nutzer-ID: safe-sp1-1446106868889-015945585
PAIR Finance GmbH
vertr.d.d. GF
Knesebeckstraße 62-63
10719 Berlin
Mein Zeichen
Ihr Zeichen
16 DR 427/26
195691380487
Ruderting, 15.06.2026
Bitte immer angeben!
Zwangsvollstreckungssache
Octopus Energy Germany GmbH, August-Everding-Straße 25, 81671 München
vertr. d.
PAIR Finance GmbH, Knesebeckstraße 62-63, 10719 Berlin, Tel. 030/340602950, Fax 030/340602951,
E-Mail aftercourt@pairfinance.de
gegen
Frau Irmgard Dirndorfer, Reuth 33, 94538 Fürstenstein
Sehr geehrte Damen und Herren,
in obe

In [17]:
corrected_df

,gpu_task_uuid,task_uuid,ticket_uuid,model_name,status,model_input,model_output,processed_model_output,retry_count,attachment_id,meta_info,created_at,updated_at,text,corrected_is_dritt,flip_reason
0,5bd4480e-d385-4a43-879d-d12177c4ddec,368c2790-f579-4a13-aabd-1cc6a29e3df5,e340c001-5aac-5f5d-a877-83355bf5b0d9,drittauskunft_egvp,done,{'text': 'Obergerichtsvollzieherin Astrid Weiß...,"{'answer': '{""is_dritt"":true}', 'answer_reason...",{'is_dritt': True},0,68998399,"{'instance': {'private_ip': '100.2.36.225', 'i...",2026-06-17 13:22:43.425887,2026-06-17 13:41:42.226535,Obergerichtsvollzieherin Astrid Weiß\nLusenstr...,False,algo1_protokoll_no_ergebnis
1,aa358993-32be-47ee-8318-71d4a564a99c,09ecb2cc-64be-4428-b7ba-22ce94446241,5b196f19-cb65-5bc5-a044-792277bb984b,drittauskunft_egvp,done,{'text': 'STEPHAN LANDKAMMER Die nachstehend g...,"{'answer': '{""is_dritt"":true}', 'answer_reason...",{'is_dritt': True},0,68989504,"{'instance': {'private_ip': '100.2.36.1', 'ins...",2026-06-17 08:25:54.012925,2026-06-17 08:49:21.416005,STEPHAN LANDKAMMER\nDie nachstehend gewählten ...,False,algo1_protokoll_no_ergebnis
2,6038c453-4e2a-4cae-a4f8-367b50a40a01,1e1135df-0b3f-40de-a9eb-a8a7f76163ba,4db045d5-97e8-574c-98d5-ff75bf296919,drittauskunft_egvp,done,{'text': 'Barbara Ziske Die nachstehend gewähl...,"{'answer': '{""is_dritt"":true}', 'answer_reason...",{'is_dritt': True},0,68867765,"{'instance': {'private_ip': '100.2.38.122', 'i...",2026-06-16 13:26:11.722312,2026-06-16 13:47:48.606599,Barbara Ziske\nDie nachstehend gewählten Formu...,False,algo1_protokoll_no_ergebnis
3,58c31f46-a1e3-4036-822b-7f5104d5e749,8938f50c-f942-412c-a4ed-bf04ba039df2,6c1f49f3-fba0-5793-b3d6-e35bbedc1461,drittauskunft_egvp,done,{'text': 'Obergerichtsvollzieher Gutfleischstr...,"{'answer': '{""is_dritt"":true}', 'answer_reason...",{'is_dritt': True},0,68856476,"{'instance': {'private_ip': '100.2.36.1', 'ins...",2026-06-16 06:12:31.343785,2026-06-16 06:41:53.634661,Obergerichtsvollzieher\nGutfleischstraße 1\nCh...,False,algo1_protokoll_no_ergebnis
4,fc3336a1-d574-43dd-bc13-e26ee0d2154e,ef53b029-f776-462e-a019-57daf5ad6b2c,a621ade7-9539-5600-8e20-fa36da3c72b8,drittauskunft_egvp,done,{'text': 'Obergerichtsvollzieherin c/o AG Herf...,"{'answer': '{""is_dritt"":true}', 'answer_reason...",{'is_dritt': True},0,68705724,"{'instance': {'private_ip': '100.2.38.65', 'in...",2026-06-15 11:25:01.240759,2026-06-15 11:56:21.020679,"Obergerichtsvollzieherin\nc/o AG Herford, Auf ...",False,algo1_protokoll_no_ergebnis
5,b04e6677-d820-4295-a4df-ad56215ae674,e3b71426-a33d-4579-ae9e-0287d357bf2e,f182c48d-7a38-5cd3-b805-22214ad3b0a7,drittauskunft_egvp,done,{'text': 'Gerichtsvollzieherin Bismarckstraße ...,"{'answer': '{""is_dritt"":true}', 'answer_reason...",{'is_dritt': True},0,68334592,"{'instance': {'private_ip': '100.2.36.225', 'i...",2026-06-12 13:24:26.026738,2026-06-12 13:34:10.586081,Gerichtsvollzieherin\nBismarckstraße 23\nN. El...,False,algo1_protokoll_no_ergebnis
6,4863b1fc-cb1d-4e5a-ba2b-d9af5e7c145a,8572b1cb-b9d6-4a45-8b9d-bad8eec97565,04066a6d-3507-5db0-8a96-8f4204e51a8f,drittauskunft_egvp,done,{'text': 'Gerichtsvollzieherin Bismarckstraße ...,"{'answer': '{""is_dritt"":true}', 'answer_reason...",{'is_dritt': True},0,68309623,"{'instance': {'private_ip': '100.2.36.225', 'i...",2026-06-11 13:23:23.433344,2026-06-11 13:27:57.156833,Gerichtsvollzieherin\nBismarckstraße 23\nS. Ka...,False,algo1_protokoll_no_ergebnis
7,c9e55333-7563-469d-9630-fc6491817275,d5169b3b-a4e5-4068-822d-899e6e32511b,c9cbbbde-db1d-5871-9b08-082015944c97,drittauskunft_egvp,done,{'text': 'Daniel Amelang Hainhölzer Straße 5 O...,"{'answer': '{""is_dritt"":true}', 'answer_reason...",{'is_dritt': True},0,68301088,"{'instance': {'private_ip': '100.2.36.1', 'ins...",2026-06-11 06:23:07.987766,2026-06-11 06:54:17.373670,Daniel Amelang\nHainhölzer Straße 5\nObergeric...,False,algo2_coverletter_no_ergebnis
8,ba28224a-a5de-40b1-9a2c-4717cfe24849,622af3bc-1dcc-497c-9a42-ceac91662338,31593dcd-2749-5e4e-b166-f229ebe1c34

In [18]:
# Download PDFs for all corrected (flipped) cases.
import boto3
from IPython.display import clear_output
from python_utilities.db_connection import DbConnection

from utils.prod_utils import get_data_by_attachment_id

DOWNLOAD_DIR = "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/pdfs/tmp_dritt_corrected_check"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

analytics_db = DbConnection("ANALYTICS", "PROD_RDS")
_boto_session = boto3.Session(profile_name="739275445236_DataScienceUser")
s3 = _boto_session.client("s3")

attachment_ids = corrected_df["attachment_id"].dropna().unique().tolist()
print(f"Downloading PDFs for {len(attachment_ids)} corrected attachment(s) → {DOWNLOAD_DIR}\n")

for i, a_id in enumerate(attachment_ids, 1):
    get_data_by_attachment_id(
        a_id,
        analytics_db,
        s3,
        pdf_download=True,
        pdf_download_dir=DOWNLOAD_DIR,
        verbose=False,
    )
    clear_output(wait=True)
    print(f"[{i}/{len(attachment_ids)}] downloaded attachment_id={a_id}")

print("\nDone.")


[19/19] downloaded attachment_id=67495311

Done.
